# Toy Experiment: Discrete Diffusion on Graphs (Node Features Only)

This notebook extends `toy_experiment_discrete.ipynb` to the **graph setting**.
The key change is replacing the CNN with a **Graph Neural Network (GNN)**
that operates on an explicit graph representation of the image.

## What changes vs. the discrete CNN notebook
| Component | Discrete CNN | This notebook (Graph) |
|---|---|---|
| Data representation | Image tensor [B, 1, H, W] | Graph: node features [N, 1] + edge_index [2, E] |
| Denoising network | MinimalResNet (Conv2d) | GraphResNet (manual message passing) |
| Spatial inductive bias | Square receptive field (conv kernel) | Hop-based receptive field (GNN layers) |
| Boundary handling | Circular padding | Periodic edge_index |

## What stays the same
- D3PM-uniform forward process (transition matrix, cosine schedule)
- x₀-parameterisation and binary cross-entropy training loss
- D3PM reverse sampling
- Local consistency evaluation
- Black/white toy dataset

## Why GNNs for graphs?
CNNs assume inputs live on a **regular grid** (pixels). Graphs are **irregular** —
nodes can have different numbers of neighbors. GNNs generalise convolution to
arbitrary graph structure via **message passing**: each node aggregates information
from its neighbors, regardless of how many there are or how they are arranged.

On this toy (a regular pixel grid), GNN and CNN are equivalent in expressiveness.
The point is to establish the GNN infrastructure that will later generalise
to irregular molecule graphs (QM9).

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import csv, os

repo_dir = Path('convolutional_diffusion')
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# ── Experiment config ──────────────────────────────────────────────────────────
NUM_LAYERS_LIST = [10, 12, 16]  # next depths to test (1,2,4,8 already logged)
HIDDEN_DIM      = 256           # hidden dimension per node (matches CNN emb_dim)
H = W           = 24            # image / grid size
NSTEPS_VALID    = 400           # many steps  → locally consistent
NSTEPS_INVALID  = 3             # few steps   → should show violations
# ──────────────────────────────────────────────────────────────────────────────

K  = 2      # number of categories: 0 = black, 1 = white
T  = 500    # diffusion timesteps
N  = H * W  # number of nodes in the graph
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Config: NUM_LAYERS_LIST={NUM_LAYERS_LIST}, HIDDEN_DIM={HIDDEN_DIM}, H={H}')
print(f'Graph: {N} nodes, 4-connected grid with periodic boundary')
print(f'Sampling: NSTEPS_VALID={NSTEPS_VALID}, NSTEPS_INVALID={NSTEPS_INVALID}')

# Dataset: same two binary images as the discrete CNN experiment
black = torch.zeros(1, 1, H, W, dtype=torch.long)
white = torch.ones( 1, 1, H, W, dtype=torch.long)
X = torch.cat([black, white], dim=0)
y = torch.tensor([0, 1])
dataset = TensorDataset(X, y)
print(f'Dataset: {len(dataset)} samples, image size {H}x{W}, K={K} categories')

## Graph Construction: Pixel Grid as a Graph

We represent the 24×24 image as a graph with **N = H×W = 576 nodes**.
Each pixel becomes a node with a binary feature (0 = black, 1 = white).

**Edges** connect each pixel to its 4 grid neighbors (up, down, left, right)
with **periodic (toroidal) boundary conditions** — the same as circular padding
in the CNN. This means the top row connects to the bottom row, and the left
column connects to the right column.

We store edges as an `edge_index` tensor of shape [2, E] where:
- `edge_index[0]` = target node indices (who receives the message)
- `edge_index[1]` = source node indices (who sends the message)

Each node has exactly 4 neighbors, so E = 4×N = 2304 directed edges.

**Receptive field of a k-layer GNN** on this graph:
- Layer 1: each node sees its 4 direct neighbors
- Layer 2: each node sees neighbors-of-neighbors (up to 12 nodes in a diamond)
- Layer k: all nodes within Manhattan distance k

This is analogous to the CNN receptive field but diamond-shaped instead of square.

In [ ]:
def build_grid_edge_index(H, W, device):
    """
    Build edge_index for a 4-connected H×W grid with periodic boundary.
    Node index for pixel (i, j) = i * W + j.
    Returns edge_index of shape [2, 4*H*W]:
      edge_index[0] = target nodes (message receivers)
      edge_index[1] = source nodes (message senders)
    """
    rows, cols = [], []
    for i in range(H):
        for j in range(W):
            node = i * W + j
            # 4 neighbors with periodic boundary
            neighbors = [
                ((i - 1) % H) * W + j,   # up
                ((i + 1) % H) * W + j,   # down
                i * W + (j - 1) % W,     # left
                i * W + (j + 1) % W,     # right
            ]
            for nb in neighbors:
                rows.append(node)  # target
                cols.append(nb)    # source
    edge_index = torch.tensor([rows, cols], dtype=torch.long, device=device)
    return edge_index


edge_index = build_grid_edge_index(H, W, device)
print(f'edge_index shape: {edge_index.shape}  ({edge_index.shape[1]} directed edges)')
print(f'Each node has exactly {edge_index.shape[1] // N} neighbors')

# Visualise the neighborhood of the top-left pixel (node 0)
node0_neighbors = edge_index[1][edge_index[0] == 0]
print(f'Neighbors of node 0 (pixel 0,0): {node0_neighbors.tolist()}')
print(f'  → up=({(H-1)*W}), down=(W={W}), left=({W-1}), right=(1)  [periodic]')

## D3PM Forward Process (unchanged)

The forward process is **identical** to the discrete CNN experiment — we corrupt
each pixel independently using the D3PM-uniform transition matrix.
The graph structure does not affect the forward process; only the denoising
network (which we replace with a GNN) uses the graph.

Recall the cumulative forward process:
$$q(x_t = j \mid x_0 = i) = \bar{\alpha}_t \cdot \mathbf{1}[i=j] + \frac{1 - \bar{\alpha}_t}{K}$$

Sampling is equivalent to: keep $x_0$ with probability $\bar{\alpha}_t$,
else draw uniform from $\{0, 1\}$.

In [ ]:
def cosine_alpha_bar(t, s=0.008):
    f_t = torch.cos((t + s) / (1.0 + s) * (torch.pi / 2.0)) ** 2
    f_0 = torch.cos((torch.zeros_like(t) + s) / (1.0 + s) * (torch.pi / 2.0)) ** 2
    return (f_t / f_0.clamp(min=1e-8)).clamp(0.0, 1.0)


def q_sample(x0, t_float):
    """
    Forward process: corrupt x0 -> x_t.
    x0: LongTensor [B, 1, H, W] in {0, 1}
    t_float: FloatTensor [B] in (0, 1]
    """
    ab   = cosine_alpha_bar(t_float).to(x0.device)[:, None, None, None].expand_as(x0.float())
    keep = torch.bernoulli(ab).bool()
    noise = torch.randint(0, K, x0.shape, device=x0.device)
    return torch.where(keep, x0, noise)


def q_posterior_probs(x_t, x0, t_float, t_prev_float):
    """
    D3PM posterior: P(x_{t-1} = 1 | x_t, x_0) for each pixel.
    x_t, x0: LongTensor [B, 1, H, W]
    Returns FloatTensor [B, 1, H, W]
    """
    ab_t    = cosine_alpha_bar(t_float).to(x_t.device)[:, None, None, None]
    ab_prev = cosine_alpha_bar(t_prev_float).to(x_t.device)[:, None, None, None]
    alpha_t = (ab_t / ab_prev.clamp(min=1e-8)).clamp(0.0, 1.0)
    beta_t  = 1.0 - alpha_t
    xt_f, x0_f = x_t.float(), x0.float()
    qt_c0   = (1.0 - beta_t) * (1.0 - xt_f) + (beta_t / K) * xt_f
    qt_c1   = (beta_t / K)   * (1.0 - xt_f) + (1.0 - beta_t) * xt_f
    qbar_c0 = (1.0 + ab_prev) / 2.0 * (1.0 - x0_f) + (1.0 - ab_prev) / 2.0 * x0_f
    qbar_c1 = (1.0 - ab_prev) / 2.0 * (1.0 - x0_f) + (1.0 + ab_prev) / 2.0 * x0_f
    theta0  = qt_c0 * qbar_c0
    theta1  = qt_c1 * qbar_c1
    return theta1 / (theta0 + theta1 + 1e-8)


print('Forward process functions defined.')

## GNN Architecture: Batched Manual Message Passing

We implement message passing **without PyTorch Geometric**, using plain PyTorch
index and scatter operations. The key design choice is to process the **full
batch at once** (vectorised), rather than looping over samples — this is essential
for GPU efficiency.

### One message-passing step (batched)
For a batch of B graphs, node features have shape **[B, N, F]**.

$$\mathbf{m}_i^{(b)} = \frac{1}{|\mathcal{N}(i)|} \sum_{j \in \mathcal{N}(i)} \mathbf{h}_j^{(b,\ell)}
\quad \Rightarrow \quad
\mathbf{h}_i^{(b,\ell+1)} = \mathbf{h}_i^{(b,\ell)} + \text{ReLU}\!\left(W(\mathbf{h}_i^{(b,\ell)} + \mathbf{m}_i^{(b)} + \mathbf{e}_t^{(b)})\right)$$

The aggregate step uses `scatter_add_` on the node dimension, applied identically
across all B batch elements simultaneously — no Python loop over the batch.

### Image ↔ Graph reshaping
- **Image → Graph**: [B, 1, H, W] → reshape to [B, N, 1]
- **Graph → Image**: [B, N, 1] → reshape to [B, 1, H, W]

This is a zero-copy view operation (same memory, different shape).

In [ ]:
class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding → [B, emb_dim]."""
    def __init__(self, emb_dim):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU())

    def forward(self, t):
        d     = self.emb_dim // 2
        freqs = 10000 ** (torch.arange(d, device=t.device).float() / max(d - 1, 1))
        args  = t[:, None] / freqs[None, :]
        emb   = torch.cat([torch.sin(args), torch.cos(args)], dim=1)  # [B, emb_dim]
        return self.proj(emb)


class MessagePassingLayer(nn.Module):
    """
    Batched message passing: operates on [B, N, F] node features.

    For every batch element simultaneously:
      1. Gather:    msg[b, e, :] = h[b, source[e], :]        [B, E, F]
      2. Aggregate: agg[b, i, :] = mean_{e: target[e]=i} msg [B, N, F]
      3. Update:    h = h + ReLU(Linear(h + agg + t_emb))
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(hidden_dim, hidden_dim)
        self.act    = nn.ReLU()

    def forward(self, h, edge_index, t_emb):
        """
        h:          [B, N, F]
        edge_index: [2, E]
        t_emb:      [B, F]   one embedding per batch element
        """
        B, N, F = h.shape
        E = edge_index.shape[1]
        target, source = edge_index[0], edge_index[1]

        # 1. Gather: [B, E, F]
        msg = h[:, source, :]

        # 2. Aggregate via scatter_add on node dimension
        idx = target.view(1, E, 1).expand(B, E, F)
        agg = torch.zeros(B, N, F, device=h.device)
        agg.scatter_add_(1, idx, msg)
        agg = agg / 4.0  # mean (degree = 4 everywhere on this grid)

        # 3. Residual update; broadcast t_emb from [B, F] to [B, N, F]
        t_broadcast = t_emb.unsqueeze(1).expand(B, N, F)
        delta = self.act(self.linear(h + agg + t_broadcast))
        return h + delta


class GraphResNet(nn.Module):
    """
    Graph analogue of MinimalResNet — fully batched.
      up_projection:   Linear(1 → hidden_dim)
      num_layers ×     MessagePassingLayer  +  TimestepEmbedding
      down_projection: Linear(hidden_dim → 1)
    """
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.up     = nn.Linear(1, hidden_dim)
        self.t_embs = nn.ModuleList([TimestepEmbedding(hidden_dim) for _ in range(num_layers)])
        self.layers = nn.ModuleList([MessagePassingLayer(hidden_dim) for _ in range(num_layers)])
        self.down   = nn.Linear(hidden_dim, 1)

    def forward(self, t, x_nodes, edge_index):
        """
        t:          [B]       timestep per sample
        x_nodes:    [B, N, 1] normalised node features
        edge_index: [2, E]    fixed graph topology
        Returns:    [B, N, 1] logits per node
        """
        h = self.up(x_nodes)   # [B, N, hidden_dim]

        for layer, t_emb_mod in zip(self.layers, self.t_embs):
            t_emb = t_emb_mod(t)          # [B, hidden_dim]
            h = layer(h, edge_index, t_emb)

        return self.down(h)    # [B, N, 1]


class DiscreteDiffusionGNN(nn.Module):
    """
    GNN for discrete (K=2) diffusion using the x0-parameterisation.
    Wraps GraphResNet and handles image <-> graph reshaping so the
    training and sampling code is identical to the CNN version.
    """
    def __init__(self, H, W, hidden_dim, num_layers, edge_index):
        super().__init__()
        self.H = H
        self.W = W
        self.N = H * W
        self.register_buffer('edge_index', edge_index)
        self.gnn = GraphResNet(hidden_dim=hidden_dim, num_layers=num_layers)

    def forward(self, t, x_img):
        """
        t:     [B]           timestep
        x_img: [B, 1, H, W]  normalised noisy image in {-1, +1}
        Returns logits [B, 1, H, W]
        """
        B = x_img.shape[0]
        x_nodes      = x_img.view(B, self.N, 1)                   # [B, N, 1]
        logits_nodes = self.gnn(t, x_nodes, self.edge_index)       # [B, N, 1]
        return logits_nodes.view(B, 1, self.H, self.W)             # [B, 1, H, W]

    def predict_x0_probs(self, t, x_t):
        """x_t: LongTensor {0,1}. Returns P(x0=1) as FloatTensor."""
        return torch.sigmoid(self.forward(t, x_t.float() * 2.0 - 1.0))

    @torch.no_grad()
    def sample(self, nsteps=200, seed=None):
        """D3PM reverse sampling."""
        if seed is not None:
            torch.manual_seed(seed)
        self.eval()
        x_t = torch.randint(0, K, (1, 1, H, W), device=device)

        for i in range(nsteps, 0, -1):
            t_float = torch.tensor([i / nsteps],       device=device)
            t_prev  = torch.tensor([(i - 1) / nsteps], device=device)
            p_x0_1  = self.predict_x0_probs(t_float, x_t)
            p_x0_0  = 1.0 - p_x0_1

            if i == 1:
                x_t = (p_x0_1 > 0.5).long()
            else:
                x0_z = torch.zeros_like(x_t)
                x0_o = torch.ones_like(x_t)
                p_prev_1 = (p_x0_0 * q_posterior_probs(x_t, x0_z, t_float, t_prev)
                          + p_x0_1 * q_posterior_probs(x_t, x0_o, t_float, t_prev))
                x_t = torch.bernoulli(p_prev_1).long()

        return x_t


print('GNN classes defined.')

## Training Function

Training is **identical** to the discrete CNN experiment:
1. Sample a random timestep $t \sim \text{Uniform}\{1, \ldots, T\}$.
2. Corrupt $x_0 \to x_t$ via the forward process.
3. Ask the GNN to predict $\tilde{p}_\theta(x_0 = 1 \mid x_t, t)$ for each node.
4. Minimise binary cross-entropy against the true $x_0$.

The function is defined here; it is **called inside the experiment loop** below
for each value of `NUM_LAYERS`.

In [ ]:
def train_discrete_diffusion(model, dataset, num_epochs=300, lr=2e-4, nsteps=T):
    sampler   = RandomSampler(dataset, replacement=True, num_samples=20000)
    loader    = DataLoader(dataset, batch_size=64, sampler=sampler)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    final_loss = 0.0

    for epoch in range(num_epochs):
        total_loss = 0.0
        for x0_batch, _ in loader:
            x0_batch = x0_batch.to(device)
            B = x0_batch.shape[0]

            t_int   = torch.randint(1, nsteps + 1, (B,), device=device)
            t_float = t_int.float() / nsteps
            x_t     = q_sample(x0_batch, t_float)
            x_norm  = x_t.float() * 2.0 - 1.0         # {0,1} -> {-1,+1}
            logits  = model(t_float, x_norm)

            loss = F.binary_cross_entropy_with_logits(logits, x0_batch.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        final_loss = total_loss / len(loader)
        if (epoch + 1) % 50 == 0:
            print(f'  Epoch {epoch+1:>3}/{num_epochs}  loss={final_loss:.4f}')

    model.eval()
    return model, final_loss


print('Training function defined.')

## Experiment Sweep: Varying GNN Depth

We now sweep over `NUM_LAYERS ∈ {1, 2, 4, 8}` to find the **critical receptive field**
at which the GNN collapses — analogous to the CNN sweep where we found that
`NUM_LAYERS=8` (RF=21×21) collapsed but `NUM_LAYERS=2` (RF=9×9) did not.

For a GNN on the 4-connected pixel grid, the receptive field is a **diamond** with
Manhattan radius equal to `NUM_LAYERS`:

| NUM_LAYERS | RF radius | Nodes in RF | % of 576 nodes |
|---|---|---|---|
| 1 | 1 hop | ~5 | 0.9% |
| 2 | 2 hops | ~13 | 2.3% |
| 4 | 4 hops | ~41 | 7.1% |
| 8 | 8 hops | ~145 | 25% |

Because the GNN RF is diamond-shaped and grows quadratically with depth,
we expect collapse to happen at a **much higher** `NUM_LAYERS` than the CNN
(which collapses already at `NUM_LAYERS=8` on a 24×24 grid).

**Evaluation**: for each depth, we generate 4 samples with many steps
(`NSTEPS_VALID=400`, should be locally consistent) and 4 samples with
very few steps (`NSTEPS_INVALID=3`, should show violations when the model
has locality). Red pixels = violations of 3×3 local consistency.

In [ ]:
def local_consistency_mask(binary_img):
    binary_img = binary_img.cpu()
    h, w = binary_img.shape
    bad = torch.zeros((h, w), dtype=torch.bool)
    for i in range(h):
        rows = torch.arange(i - 1, i + 2) % h
        for j in range(w):
            cols = torch.arange(j - 1, j + 2) % w
            patch = binary_img[rows][:, cols]
            majority = 1 if patch.sum().item() >= 5 else 0
            bad[i, j] = (majority != binary_img[i, j].item())
    return bad


def draw_binary(ax, binary_img, violation_mask=None):
    img_np = binary_img.detach().cpu().float().numpy()
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_xticks([x - 0.5 for x in range(binary_img.shape[1] + 1)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(binary_img.shape[0] + 1)], minor=True)
    ax.grid(which='minor', color='#2b4ea2', linestyle='-', linewidth=0.4)
    if violation_mask is not None and bool(violation_mask.any()):
        overlay = torch.zeros((*binary_img.shape, 4), dtype=torch.float32)
        overlay[..., 0] = violation_mask.float()
        overlay[..., 3] = violation_mask.float() * 0.45
        ax.imshow(overlay.numpy())
    ax.set_xticks([])
    ax.set_yticks([])


training_imgs = [torch.zeros((H, W), dtype=torch.int), torch.ones((H, W), dtype=torch.int)]

print('Evaluation helpers defined.')

In [ ]:
log_path   = 'results_log_graph.csv'
fieldnames = ['model', 'num_layers', 'hidden_dim', 'image_size', 'n_params',
              'nsteps_valid', 'nsteps_invalid', 'final_loss',
              'valid_violations_mean', 'invalid_violations_mean',
              'valid_violations_all', 'invalid_violations_all']

# Read already-completed depths so we never redo work
already_done = set()
if os.path.exists(log_path):
    with open(log_path) as f:
        for row in csv.DictReader(f):
            already_done.add(int(row['num_layers']))
    print(f'Already logged NUM_LAYERS: {sorted(already_done)} — will skip these')

write_header = not os.path.exists(log_path)

for NUM_LAYERS in NUM_LAYERS_LIST:
    if NUM_LAYERS in already_done:
        print(f'Skipping NUM_LAYERS={NUM_LAYERS} (already in log)')
        continue

    print(f"\n{'='*60}")
    print(f"Experiment  NUM_LAYERS={NUM_LAYERS}  "
          f"(receptive field = {NUM_LAYERS}-hop diamond)")
    print(f"{'='*60}")

    # ── Build model ────────────────────────────────────────────
    model = DiscreteDiffusionGNN(
        H=H, W=W, hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS, edge_index=edge_index,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {n_params:,}')

    # ── Train ──────────────────────────────────────────────────
    model, final_loss = train_discrete_diffusion(
        model, dataset, num_epochs=300, lr=2e-4, nsteps=T)
    print(f'Training complete. Final loss: {final_loss:.4f}')

    # ── Generate & evaluate ────────────────────────────────────
    valid_samples   = [model.sample(nsteps=NSTEPS_VALID)[0, 0].cpu()   for _ in range(4)]
    invalid_samples = [model.sample(nsteps=NSTEPS_INVALID)[0, 0].cpu() for _ in range(4)]
    valid_masks     = [local_consistency_mask(img) for img in valid_samples]
    invalid_masks   = [local_consistency_mask(img) for img in invalid_samples]

    valid_counts   = [int(m.sum()) for m in valid_masks]
    invalid_counts = [int(m.sum()) for m in invalid_masks]
    print(f'Valid violations   ({NSTEPS_VALID} steps):   {valid_counts}  mean={np.mean(valid_counts):.2f}')
    print(f'Invalid violations ({NSTEPS_INVALID} steps): {invalid_counts}  mean={np.mean(invalid_counts):.2f}')

    # ── Plot ───────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 4))
    outer = GridSpec(1, 3, width_ratios=[2, 4, 4], wspace=0.15)

    left = outer[0].subgridspec(1, 2, wspace=0.1)
    ax_a0 = fig.add_subplot(left[0, 0])
    ax_a1 = fig.add_subplot(left[0, 1])
    draw_binary(ax_a0, training_imgs[0])
    draw_binary(ax_a1, training_imgs[1])
    ax_a0.set_title('Training Data', fontsize=12, pad=10)

    mid = outer[1].subgridspec(1, 4, wspace=0.1)
    mid_axes = [fig.add_subplot(mid[0, i]) for i in range(4)]
    for ax, img in zip(mid_axes, valid_samples):
        draw_binary(ax, img)
    mid_axes[1].set_title(f'Valid Samples ({NSTEPS_VALID} steps)', fontsize=11, pad=10)

    right = outer[2].subgridspec(1, 4, wspace=0.1)
    right_axes = [fig.add_subplot(right[0, i]) for i in range(4)]
    for ax, img, mask in zip(right_axes, invalid_samples, invalid_masks):
        draw_binary(ax, img, mask)
    right_axes[1].set_title(f'Invalid Samples ({NSTEPS_INVALID} steps)', fontsize=11, pad=10)

    fig.suptitle(f'GNN  L={NUM_LAYERS}  ({NUM_LAYERS}-hop RF)  |  '
                 f'valid mean={np.mean(valid_counts):.1f}  '
                 f'invalid mean={np.mean(invalid_counts):.1f}', fontsize=12, y=1.02)

    plt.tight_layout()
    fname = f'toy_graph_L{NUM_LAYERS}_valid{NSTEPS_VALID}_invalid{NSTEPS_INVALID}.png'
    plt.savefig(fname, dpi=180, bbox_inches='tight')
    print(f'Saved {fname}')
    plt.show()

    # ── Log ────────────────────────────────────────────────────
    row = {
        'model':                   'GNN',
        'num_layers':              NUM_LAYERS,
        'hidden_dim':              HIDDEN_DIM,
        'image_size':              H,
        'n_params':                n_params,
        'nsteps_valid':            NSTEPS_VALID,
        'nsteps_invalid':          NSTEPS_INVALID,
        'final_loss':              round(final_loss, 4),
        'valid_violations_mean':   round(np.mean(valid_counts),   2),
        'invalid_violations_mean': round(np.mean(invalid_counts), 2),
        'valid_violations_all':    str(valid_counts),
        'invalid_violations_all':  str(invalid_counts),
    }
    with open(log_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
            write_header = False
        writer.writerow(row)
    already_done.add(NUM_LAYERS)

print(f"\n{'='*60}")
print(f"All experiments done. Results in {log_path}")
print(f"{'='*60}")
with open(log_path) as f:
    for line in f:
        print(line, end='')